# CI smoke test: cached mode of the Ascon glitch-leakage notebook

```
SPDX-License-Identifier: Apache-2.0
```

This small notebook checks that the main notebook's helper, `setup_env.py`, works where no
circuit tools are installed. The organizers' CI installs only `pytest`, `nbmake`, `pandas`,
`graphviz` and `matplotlib`. In that case the helper switches to **cached mode**: it runs no
ngspice and needs no PDK, and it plots the precomputed results in `data/`, which
`make_cached_data.py` exports from the SPICE campaigns. The main notebook uses the same helper.
With ngspice and the sky130 PDK present (Colab installs them in about a minute), it runs live
instead.

In [ ]:
import os
import subprocess
import sys

# The helper sits next to this notebook. On Colab only the notebook is
# opened, so fetch the submission folder first (as the main notebook does).
UPSTREAM = "https://github.com/sscs-ose/sscs-ose-code-a-chip.github.io"
FOLDER = "ISSCC27/submitted_notebooks/ascon_glitch_leakage"
if not os.path.exists("setup_env.py"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none",
                    "--sparse", UPSTREAM, "code-a-chip"], check=True)
    subprocess.run(["git", "-C", "code-a-chip", "sparse-checkout", "set",
                    FOLDER], check=True)
    os.chdir(os.path.join("code-a-chip", FOLDER))
sys.path.insert(0, os.getcwd())

import setup_env

env = setup_env.setup(mode="cached")
assert env.mode == "cached"

Every cached file is listed in `data/MANIFEST.csv` with its size, SHA-256, source and
meaning. The next cell checks the files against it.

In [ ]:
import pandas as pd

manifest = pd.read_csv(setup_env.data_path("MANIFEST.csv"))
assert setup_env.check_data() == []
print("%d files, %.2f MB" % (len(manifest), manifest["bytes"].sum() / 1e6))
manifest[["file", "bytes", "description"]].head(8)

## Headline result from the cached data

These are the fixed-vs-random TVLA results of the transistor-level (ngspice, sky130 `tt`)
campaigns. They compare the unmasked S-box (U), naive masking (N), masking with a register
barrier (D) and the barrier placed after Ascon's affine layer (DA). A |t| above 4.5 means
first-order leakage.

In [ ]:
camp = pd.read_csv(setup_env.data_path("campaigns.csv"))
cols = ["campaign", "variant", "traces", "spice_max_abs_t",
        "spice_first_above_4p5", "functional_mismatches"]
camp[cols]

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for name, color in (("N_tvla", "tab:red"), ("D_tvla", "tab:orange"),
                    ("DA_tvla", "tab:green")):
    tc = setup_env.read_csv("tcurve_%s.csv" % name)
    axes[0].plot(tc["ns_after_edge"], tc["t_spice"], color=color, lw=0.8,
                 label=name.split("_")[0])
    mt = setup_env.read_csv("maxt_vs_traces_%s.csv" % name)
    axes[1].semilogx(mt["traces"], mt["spice"], color=color,
                     label=name.split("_")[0] + " SPICE")
mt = setup_env.read_csv("maxt_vs_traces_N_tvla.csv")
axes[1].semilogx(mt["traces"], mt["level1_weighted"], "k--",
                 label="N zero-delay model")
for ax in axes:
    ax.axhline(4.5, color="gray", ls=":")
axes[0].axhline(-4.5, color="gray", ls=":")
axes[0].set(xlabel="ns after the capturing clock edge", ylabel="t",
            title="Welch t per 10 ps sample (SPICE)")
axes[1].set(xlabel="traces", ylabel="max |t|",
            title="The model says safe; SPICE says N leaks")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
fig.tight_layout()

In [ ]:
row = camp.set_index("campaign")["spice_max_abs_t"]
assert row["N_tvla"] > 4.5 and row["DA_tvla"] < 4.5
keys, times, t = setup_env.read_matrix("tvla_t_checkpoints_N_tvla.csv")
assert t.shape == (len(keys["traces"]), len(times))
print("cached-mode smoke test passed")